In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

MODEL = "gemini-2.0-flash"
openai = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta", api_key=api_key)

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "¿Cuánto son 2 + 2?"}]
)

print(response.choices[0].message.content)

2 + 2 son 4.



In [3]:
# !pip install selenium pandas

import pandas as pd
import time
import re
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

print("✅ Librerías cargadas correctamente.")

✅ Librerías cargadas correctamente.


In [ ]:
def scrape_autofesa_pro(max_pages=None): 
    """
    Scraper de Autofesa que extrae TODO el catálogo.
    - FILTRA: Ignora coches con clase 'sold' (vendido) o 'on-prepare' (en preparación).
    - Datos limpios: Precio (int), Km (int), Año (int).
    """
    
    # --- CONFIGURACIÓN DEL DRIVER ---
    options = Options()
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    # options.add_argument('--headless') # Descomenta para modo silencioso
    
    driver = webdriver.Chrome(options=options)
    resultados = []
    
    print(f"🚀 Iniciando scraping COMPLETO (Filtrando vendidos/preparación)...")
    
    try:
        url = "https://www.autofesa.com/coches-segunda-mano"
        driver.get(url)
        wait = WebDriverWait(driver, 10)

        # --- COOKIES ---
        try:
            cookie_btn = WebDriverWait(driver, 4).until(
                EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
            )
            cookie_btn.click()
            print("🍪 Cookies aceptadas.")
            time.sleep(1)
        except:
            print("ℹ️ No se requirió aceptar cookies o ya estaban aceptadas.")

        # --- BUCLE DE PÁGINAS ---
        page_num = 1
        
        while True:
            if max_pages and page_num > max_pages:
                print(f"🛑 Límite manual de páginas ({max_pages}) alcanzado.")
                break
                
            print(f"\n📄 Procesando página {page_num}...")
            
            try:
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".vehicle-list__item")))
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
                time.sleep(1)
                
                car_elements = driver.find_elements(By.CSS_SELECTOR, ".vehicle-list__item")
                print(f"   ✓ Encontrados {len(car_elements)} elementos (procesando activos)...")

                coches_guardados_pagina = 0

                for car in car_elements:
                    try:
                        # === 🛡️ FILTRO NUEVO: DETECTAR VENDIDOS O EN PREPARACIÓN ===
                        # Buscamos el div interno que tiene las clases
                        try:
                            card_div = car.find_element(By.CSS_SELECTOR, ".vehicle-card")
                            clases = card_div.get_attribute("class") # Obtenemos el texto de las clases
                            
                            if "vehicle-card--sold" in clases:
                                # print("     🚫 Saltando coche VENDIDO")
                                continue
                            if "vehicle-card--on-prepare" in clases:
                                # print("     🚫 Saltando coche EN PREPARACIÓN")
                                continue
                        except:
                            # Si no encuentra la tarjeta dentro del item, saltamos por seguridad
                            continue
                        # ============================================================

                        # 1. Título
                        try:
                            title = car.find_element(By.CSS_SELECTOR, ".vehicle-card__title").text.strip()
                        except: title = "Desconocido"

                        # 2. Precio
                        try:
                            price_text = car.find_element(By.CSS_SELECTOR, ".vehicle-card__price").text
                            price_clean = int(re.sub(r'[^\d]', '', price_text))
                        except: price_clean = 0 

                        # 3. Datos Técnicos
                        km_clean = 0
                        year_clean = 0
                        fuel = "Otros"
                        
                        try:
                            features = car.find_elements(By.CSS_SELECTOR, ".vehicle-card__features .list .item")
                            for f in features:
                                txt = f.text.strip()
                                if "km" in txt.lower():
                                    km_clean = int(re.sub(r'[^\d]', '', txt))
                                elif txt.isdigit() and len(txt) == 4:
                                    year_clean = int(txt)
                                elif txt in ["Diésel", "Gasolina", "Híbrido", "Eléctrico", "GLP"]:
                                    fuel = txt
                        except: pass

                        # 4. Link e Imagen
                        try:
                            link = car.find_element(By.CSS_SELECTOR, "a").get_attribute("href")
                        except: link = "N/D"
                        
                        try:
                            img_elem = car.find_element(By.CSS_SELECTOR, "img")
                            img_src = img_elem.get_attribute("src")
                            if not img_src or "base64" in img_src:
                                img_src = img_elem.get_attribute("data-src")
                        except: img_src = "No img"

                        # Guardar solo si ha pasado el filtro
                        resultados.append({
                            "Modelo": title,
                            "Precio": price_clean,
                            "Año": year_clean,
                            "Km": km_clean,
                            "Combustible": fuel,
                            "Imagen": img_src,
                            "Link": link
                        })
                        coches_guardados_pagina += 1

                    except Exception as e:
                        continue 

                # --- SIGUIENTE PÁGINA ---
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(1)
                
                try:
                    next_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.page-link[title="Página siguiente"]')))
                    driver.execute_script("arguments[0].click();", next_button)
                    page_num += 1
                    time.sleep(2) 
                except TimeoutException:
                    print("\n🏁 FINAL: No se encontró botón 'Siguiente'.")
                    break
                    
            except Exception as e:
                print(f"❌ Error en la página {page_num}: {e}")
                break
                
    except Exception as e:
        print(f"❌ Error general del driver: {e}")
        
    finally:
        driver.quit()
        return resultados

In [10]:
def scrape_autofesa_pro(max_pages=None): 
    """
    Scraper de Autofesa que extrae TODO el catálogo.
    - FILTRA: Ignora coches con clase 'sold' (vendido) o 'on-prepare' (en preparación).
    - Datos limpios: Precio (int), Km (int), Año (int).
    """
    
    # --- CONFIGURACIÓN DEL DRIVER ---
    options = Options()
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    # options.add_argument('--headless') # Descomenta para modo silencioso
    
    driver = webdriver.Chrome(options=options)
    resultados = []
    
    print(f"🚀 Iniciando scraping COMPLETO (Filtrando vendidos/preparación)...")
    
    try:
        url = "https://www.autofesa.com/coches-segunda-mano"
        driver.get(url)
        wait = WebDriverWait(driver, 10)

        # --- COOKIES ---
        try:
            cookie_btn = WebDriverWait(driver, 4).until(
                EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
            )
            cookie_btn.click()
            print("🍪 Cookies aceptadas.")
            time.sleep(1)
        except:
            print("ℹ️ No se requirió aceptar cookies o ya estaban aceptadas.")

        # --- BUCLE DE PÁGINAS ---
        page_num = 1
        
        while True:
            if max_pages and page_num > max_pages:
                print(f"🛑 Límite manual de páginas ({max_pages}) alcanzado.")
                break
                
            print(f"\n📄 Procesando página {page_num}...")
            
            try:
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".vehicle-list__item")))
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
                time.sleep(1)
                
                car_elements = driver.find_elements(By.CSS_SELECTOR, ".vehicle-list__item")
                print(f"   ✓ Encontrados {len(car_elements)} elementos (procesando activos)...")

                coches_guardados_pagina = 0

                for car in car_elements:
                    try:
                        # === 🛡️ FILTRO NUEVO: DETECTAR VENDIDOS O EN PREPARACIÓN ===
                        # Buscamos el div interno que tiene las clases
                        try:
                            card_div = car.find_element(By.CSS_SELECTOR, ".vehicle-card")
                            clases = card_div.get_attribute("class") # Obtenemos el texto de las clases
                            
                            if "vehicle-card--sold" in clases:
                                # print("     🚫 Saltando coche VENDIDO")
                                continue
                            if "vehicle-card--on-prepare" in clases:
                                # print("     🚫 Saltando coche EN PREPARACIÓN")
                                continue
                        except:
                            # Si no encuentra la tarjeta dentro del item, saltamos por seguridad
                            continue
                        # ============================================================

                        # 1. Título
                        try:
                            title = car.find_element(By.CSS_SELECTOR, ".vehicle-card__title").text.strip()
                        except: title = "Desconocido"

                        # 2. Precio
                        try:
                            price_text = car.find_element(By.CSS_SELECTOR, ".vehicle-card__price").text
                            price_clean = int(re.sub(r'[^\d]', '', price_text))
                        except: price_clean = 0 

                        # 3. Datos Técnicos
                        km_clean = 0
                        year_clean = 0
                        fuel = "Otros"
                        
                        try:
                            features = car.find_elements(By.CSS_SELECTOR, ".vehicle-card__features .list .item")
                            for f in features:
                                txt = f.text.strip()
                                if "km" in txt.lower():
                                    km_clean = int(re.sub(r'[^\d]', '', txt))
                                elif txt.isdigit() and len(txt) == 4:
                                    year_clean = int(txt)
                                elif txt in ["Diésel", "Gasolina", "Híbrido", "Eléctrico", "GLP"]:
                                    fuel = txt
                        except: pass

                        # 4. Link e Imagen
                        try:
                            link = car.find_element(By.CSS_SELECTOR, "a").get_attribute("href")
                        except: link = "N/D"
                        
                        try:
                            img_elem = car.find_element(By.CSS_SELECTOR, "img")
                            img_src = img_elem.get_attribute("src")
                            if not img_src or "base64" in img_src:
                                img_src = img_elem.get_attribute("data-src")
                        except: img_src = "No img"

                        # Guardar solo si ha pasado el filtro
                        resultados.append({
                            "Modelo": title,
                            "Precio": price_clean,
                            "Año": year_clean,
                            "Km": km_clean,
                            "Combustible": fuel,
                            "Imagen": img_src,
                            "Link": link
                        })
                        coches_guardados_pagina += 1

                    except Exception as e:
                        continue 

                # --- SIGUIENTE PÁGINA ---
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(1)
                
                try:
                    next_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.page-link[title="Página siguiente"]')))
                    driver.execute_script("arguments[0].click();", next_button)
                    page_num += 1
                    time.sleep(2) 
                except TimeoutException:
                    print("\n🏁 FINAL: No se encontró botón 'Siguiente'.")
                    break
                    
            except Exception as e:
                print(f"❌ Error en la página {page_num}: {e}")
                break
                
    except Exception as e:
        print(f"❌ Error general del driver: {e}")
        
    finally:
        driver.quit()
        return resultados

In [ ]:
# 1. Ejecutar Scraper SIN LÍMITES (Esto puede tardar varios minutos dependiendo de la web)
print("▶️ Ejecutando scraper masivo...")

# max_pages=None significa que irá hasta el final
datos_coches = scrape_autofesa_pro(max_pages=None) 

# 2. Guardar
if datos_coches:
    df = pd.DataFrame(datos_coches)
    
    print(f"\n✅ Extracción total completada. Total coches: {len(df)}")
    display(df.tail()) # Muestra los últimos para ver si llegó al final
    
    nombre_archivo = f"autofesa_completo_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    df.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
    print(f"💾 Base de datos completa guardada en: {nombre_archivo}")
    
else:
    print("⚠️ No se han extraído datos.")

There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


▶️ Ejecutando scraper masivo...
🚀 Iniciando scraping COMPLETO (Filtrando vendidos/preparación)...
ℹ️ No se requirió aceptar cookies o ya estaban aceptadas.

📄 Procesando página 1...
   ✓ Encontrados 30 elementos (procesando activos)...

📄 Procesando página 2...
   ✓ Encontrados 30 elementos (procesando activos)...

📄 Procesando página 3...
   ✓ Encontrados 30 elementos (procesando activos)...

📄 Procesando página 4...
   ✓ Encontrados 30 elementos (procesando activos)...

📄 Procesando página 5...
   ✓ Encontrados 30 elementos (procesando activos)...

📄 Procesando página 6...
   ✓ Encontrados 30 elementos (procesando activos)...

📄 Procesando página 7...
   ✓ Encontrados 30 elementos (procesando activos)...

📄 Procesando página 8...
   ✓ Encontrados 30 elementos (procesando activos)...

📄 Procesando página 9...
   ✓ Encontrados 30 elementos (procesando activos)...

📄 Procesando página 10...
   ✓ Encontrados 30 elementos (procesando activos)...

📄 Procesando página 11...
   ✓ Encontrados